# Activation Functions — Exercises

Companion to the note [Activation Functions](Activation%20Functions.md).

Practise the activation functions from the note's table (sigmoid, tanh, ReLU, Leaky ReLU, ELU/SELU, GELU, Swish, softmax):
their derivatives, saturation and vanishing gradients, dying ReLUs, numerical stability, and how the choice of
activation interacts with initialisation and gradient flow in deep networks. All code is NumPy; references come from SciPy.

**15 exercises** · 🧠 Concept ×4 · ✍️ By hand ×5 · 💻 Code ×6

**How to use this notebook**
- Difficulty: ★☆☆ warm-up · ★★☆ standard · ★★★ challenge.
- ✍️ *By hand* exercises are meant for pen and paper (or a markdown cell). 💻 *Code* exercises have a cell for you to fill in.
- Every exercise has a folded **💡 Hint** and **✅ Solution**. Click to unfold, but try first!
- The notebook runs top to bottom as-is; `check(...)` lines print ⏳ until you have an answer, then ✅ or ❌.


In [ ]:
import numpy as np
np.random.seed(0)

# --- helper: compares your answer with a reference, prints ✅/❌, never raises ---
import numpy as _np
def check(name, got, expected, tol=1e-6):
    if got is None or got is Ellipsis:
        print(f"⏳ {name}: not attempted yet")
        return
    try:
        ok = bool(_np.allclose(_np.asarray(got, dtype=float), _np.asarray(expected, dtype=float), atol=tol, rtol=1e-4))
    except Exception:
        try:
            ok = bool(got == expected)
        except Exception:
            ok = False
    print(("✅ " if ok else "❌ ") + name + ("" if ok else f"  (got {got!r})"))

import warnings
from scipy.special import expit, erf, softmax as _sp_softmax
from scipy.stats import norm as _norm
from sklearn.datasets import make_moons
rng = np.random.default_rng(0)

def _fd(f, z, eps=1e-6):
    # reference: element-wise central-difference derivative
    return (f(z + eps) - f(z - eps)) / (2 * eps)

## Part A · Concepts

### Exercise 1 · Which activations saturate?
*🧠 Concept · ★☆☆*

An activation *saturates* where $\phi'(z)\to0$. For sigmoid, tanh, ReLU, Leaky ReLU, ELU and GELU, say where (if anywhere) each saturates
and what that means for [[Backpropagation]].

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Look at the limits $z\to+\infty$ and $z\to-\infty$ of each derivative.

</details>

<details>
<summary><b>✅ Solution</b></summary>

| Activation | $z\to-\infty$ | $z\to+\infty$ |
|---|---|---|
| sigmoid | saturates ($\phi'\to0$) | saturates |
| tanh | saturates | saturates |
| ReLU | $\phi' = 0$ exactly (dead region) | $\phi'=1$, never saturates |
| Leaky ReLU | $\phi'=\alpha>0$ | $\phi'=1$ |
| ELU | saturates smoothly to $-\alpha$ ($\phi'\to0$) | $\phi'=1$ |
| GELU | $\phi'\to0$ (output $\to 0$) | $\phi'\to1$ |

In a saturated region the factor $\phi'(z)$ in $\delta^{(l)} = (W^\top\delta^{(l+1)})\odot\phi'(z^{(l)})$ is $\approx0$, so the unit (and everything below it)
gets almost no gradient. Two-sided saturation (sigmoid/tanh) is the classic cause of vanishing gradients; the ReLU family keeps $\phi' = 1$ on the positive side.

</details>

### Exercise 2 · Dying ReLU
*🧠 Concept · ★☆☆*

1. What is a "dead" ReLU unit and how can a large learning-rate step create one?
2. Why can it not recover by itself?
3. Name three remedies.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

A unit is dead if $z = w^\top x + b < 0$ for **every** training input. What is $\partial L/\partial w$ then?

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. A unit whose pre-activation is negative for all inputs always outputs 0. A big update can push $b$ (or $w$) far negative, e.g. after a large gradient spike.
2. Its gradient is $\delta\,x^\top$ with $\delta = (\dots)\odot\mathbb 1[z>0] = 0$ for every sample: no gradient, so the weights never change again.
3. Leaky ReLU / PReLU (small slope $\alpha$ on the negative side), ELU/GELU/Swish (smooth, non-zero gradient for $z<0$), a smaller learning rate or warm-up,
   careful init (He, small positive or zero bias), normalisation layers keeping $z$ centred ([[Training Tricks]]).

</details>

### Exercise 3 · Why zero-centred outputs help
*🧠 Concept · ★★☆*

The note says tanh is *zero-centred* while sigmoid is not. Consider a unit $z = \sum_i w_ih_i$ whose inputs $h_i$ come from sigmoid units, so $h_i>0$.
Show that all components of $\partial L/\partial w$ have the **same sign** for a given example, and explain why that slows gradient descent.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

$\partial L/\partial w_i = \delta\,h_i$ with a scalar $\delta = \partial L/\partial z$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\partial L/\partial w_i = \delta h_i$ and all $h_i>0$, so every component has the sign of $\delta$. Per example the update can only move $w$ into the
"all increase" or "all decrease" orthant. If the optimum needs some weights to go up and others down, GD must zig-zag. With tanh (outputs of either sign) or
with normalisation the update directions are unconstrained. (Mini-batch averaging softens this, but it is one reason tanh beat sigmoid for hidden layers.)

</details>

### Exercise 4 · Choose the activation
*🧠 Concept · ★★☆*

Pick a sensible activation and justify briefly:

(a) Hidden layers of a 50-layer CNN with BatchNorm. (b) Hidden layers of a Transformer MLP block.
(c) The output of a model predicting a probability. (d) The gates of an LSTM. (e) A deep fully connected net **without** normalisation layers that should keep activations standardised.
(f) Why is softmax not used as a hidden-layer activation?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

The table's 'Pros/cons' column answers most of these.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) ReLU (cheap, no positive saturation; with He init). (b) GELU (standard in [[Transformers]]; Swish/SiLU-based SwiGLU in many modern LLMs).
(c) Sigmoid (or softmax for several exclusive classes). (d) Sigmoid for gates (values in $(0,1)$ act as soft switches), tanh for the candidate state ([[RNN and LSTM]]).
(e) SELU with LeCun-normal init (self-normalising, Exercise 15). (f) Softmax couples all units (they must sum to 1), so units compete and the layer has
only $K-1$ degrees of freedom; it also saturates. It is meant to turn the final logits into a distribution.

</details>

## Part B · By hand

### Exercise 5 · Sigmoid derivative
*✍️ By hand · ★☆☆*

Show $\sigma'(z) = \sigma(z)\,(1-\sigma(z))$. What is its maximum value and where? What is $\sigma'(5)$ (3 significant digits)?

In [ ]:
max_sig_grad = None
sig_grad_5 = None

_zz = np.linspace(-10, 10, 20001)
check('max of sigma\'', max_sig_grad, _fd(expit, _zz).max(), tol=1e-6)
check('sigma\'(5)', sig_grad_5, _fd(expit, 5.0), tol=1e-5)

<details>
<summary><b>💡 Hint</b></summary>

$\sigma(z) = (1+e^{-z})^{-1}$, so $\sigma' = e^{-z}(1+e^{-z})^{-2}$. Rewrite $\frac{e^{-z}}{1+e^{-z}} = 1-\sigma$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\sigma'(z) = \frac{e^{-z}}{(1+e^{-z})^2} = \frac{1}{1+e^{-z}}\cdot\frac{e^{-z}}{1+e^{-z}} = \sigma(1-\sigma)$.
$p(1-p)$ is maximised at $p=\tfrac12$, i.e. $z=0$, with value $\tfrac14$. At $z=5$: $\sigma = 0.99331$, $\sigma' = 0.00665$: already 40× smaller than the maximum.

```python
max_sig_grad = 0.25
sig_grad_5 = 0.00665
```

</details>

### Exercise 6 · tanh is a rescaled sigmoid
*✍️ By hand · ★★☆*

1. Show $\tanh z = 2\sigma(2z) - 1$.
2. Show $\tanh'(z) = 1 - \tanh^2 z$ and give $\tanh'(0)$ and $\tanh'(1)$ (4 decimals).
3. Compare $\tanh'(0)$ with $\sigma'(0)$: why does this matter for gradient flow?

In [ ]:
tanh_grad_0 = None
tanh_grad_1 = None

check("tanh'(0)", tanh_grad_0, _fd(np.tanh, 0.0))
check("tanh'(1)", tanh_grad_1, _fd(np.tanh, 1.0), tol=1e-4)

<details>
<summary><b>💡 Hint</b></summary>

$2\sigma(2z) - 1 = \frac{2}{1+e^{-2z}} - 1 = \frac{1-e^{-2z}}{1+e^{-2z}}$; multiply by $e^z/e^z$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. $\frac{1-e^{-2z}}{1+e^{-2z}} = \frac{e^{z}-e^{-z}}{e^{z}+e^{-z}} = \tanh z$.
2. Quotient rule: $\tanh' = \frac{(e^z+e^{-z})^2 - (e^z-e^{-z})^2}{(e^z+e^{-z})^2} = 1-\tanh^2 z$. $\tanh'(0) = 1$, $\tanh'(1) = 1 - 0.7616^2 = 0.4200$.
3. From (1), $\tanh'(z) = 4\sigma'(2z)$, so at 0 the slope is 4× that of sigmoid (1 vs 0.25): a near-zero tanh layer passes gradients through
   unchanged instead of shrinking them by 4×, besides being zero-centred.

```python
tanh_grad_0 = 1.0
tanh_grad_1 = 0.4200
```

</details>

### Exercise 7 · Softmax: shift invariance and Jacobian
*✍️ By hand · ★★☆*

1. Show $\mathrm{softmax}(z + c\mathbf 1) = \mathrm{softmax}(z)$ for any scalar $c$.
2. Show $\partial p_i/\partial z_j = p_i([i=j] - p_j)$, i.e. $J = \mathrm{diag}(p) - pp^\top$.
3. Compute $J$ for $z = (0, \ln 2)$.

In [ ]:
J_hand = None   # 2x2

_z = np.array([0., np.log(2)])
_J = np.stack([_fd(lambda t: _sp_softmax(_z + t * np.eye(2)[j])[...], 0.0) for j in range(2)], axis=1)
check('Jacobian', J_hand, _J, tol=1e-4)

<details>
<summary><b>💡 Hint</b></summary>

For (3): $p = (1/3, 2/3)$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. $\frac{e^{z_i+c}}{\sum_j e^{z_j+c}} = \frac{e^ce^{z_i}}{e^c\sum_j e^{z_j}}$. This is why we can subtract $\max z$ for stability.
2. $\partial p_i/\partial z_j = \frac{[i=j]e^{z_i}S - e^{z_i}e^{z_j}}{S^2} = p_i[i=j] - p_ip_j$.
3. $p = (1/3, 2/3)$: $J = \begin{pmatrix}1/3 - 1/9 & -2/9\\ -2/9 & 2/3-4/9\end{pmatrix} = \begin{pmatrix}2/9&-2/9\\-2/9&2/9\end{pmatrix}$.
   Rows sum to 0 (shift invariance) and $J$ is singular: softmax has only $K-1$ degrees of freedom.

```python
J_hand = [[2/9, -2/9], [-2/9, 2/9]]
```

</details>

### Exercise 8 · GELU and Swish at a few points
*✍️ By hand · ★☆☆*

With $\mathrm{GELU}(z) = z\,\Phi(z)$ ($\Phi$ = standard normal CDF, $\Phi(1) = 0.8413$) and $\mathrm{Swish}(z) = z\,\sigma(z)$:
compute GELU(1), GELU(−1), Swish(2) and Swish(−2) (4 decimals). Which has a larger negative dip?

In [ ]:
gelu_1 = None
gelu_m1 = None
swish_2 = None
swish_m2 = None

check('GELU(1)', gelu_1, 1 * _norm.cdf(1), tol=1e-4)
check('GELU(-1)', gelu_m1, -1 * _norm.cdf(-1), tol=1e-4)
check('Swish(2)', swish_2, 2 * expit(2), tol=1e-4)
check('Swish(-2)', swish_m2, -2 * expit(-2), tol=1e-4)

<details>
<summary><b>💡 Hint</b></summary>

$\Phi(-1) = 1 - \Phi(1)$ and $\sigma(-2) = 1 - \sigma(2)$, with $\sigma(2) = 0.8808$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

GELU(1) $= 0.8413$, GELU(−1) $= -0.1587$; Swish(2) $= 1.7616$, Swish(−2) $= -0.2384$.
Both are smooth, non-monotone "soft ReLUs": they let small negative values through (non-zero gradient for $z<0$, so no dead units).
Minimum values: GELU $\approx -0.170$ at $z\approx-0.75$; Swish $\approx -0.278$ at $z\approx-1.28$, so Swish dips deeper.

```python
gelu_1 = 0.8413
gelu_m1 = -0.1587
swish_2 = 1.7616
swish_m2 = -0.2384
```

</details>

### Exercise 9 · Derivative of Swish
*✍️ By hand · ★★★*

Derive $\frac{d}{dz}\,z\sigma(z)$ and simplify it to $s(z) + \sigma(z)(1 - s(z))$ where $s(z) = z\sigma(z)$.
Evaluate it at $z=0$ and $z = 2$ (4 decimals). Explain why the derivative can **exceed 1**.

In [ ]:
swish_grad_0 = None
swish_grad_2 = None

_sw = lambda z: z * expit(z)
check("swish'(0)", swish_grad_0, _fd(_sw, 0.0))
check("swish'(2)", swish_grad_2, _fd(_sw, 2.0), tol=1e-4)

<details>
<summary><b>💡 Hint</b></summary>

Product rule: $\sigma(z) + z\sigma'(z) = \sigma + z\sigma(1-\sigma)$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\frac{d}{dz}z\sigma = \sigma + z\sigma(1-\sigma) = \sigma + s - s\sigma = s + \sigma(1-s)$ (handy: reuse the cached forward output).
At $0$: $0.5$. At $2$: $\sigma=0.8808$, $s = 1.7616$: $1.7616 + 0.8808\cdot(1-1.7616) = 1.0908$.
Swish approaches the identity from **below** on the positive side ($z\sigma(z) < z$), so it must rise faster than slope 1 to catch up: $\phi' > 1$ for moderate $z$ (max $\approx1.10$ at $z\approx2.4$).

```python
swish_grad_0 = 0.5
swish_grad_2 = 1.0908
```

</details>

## Part C · Code

### Exercise 10 · An activation library with derivatives
*💻 Code · ★☆☆*

Fill in the dictionary `ACT`: for each name, a pair `(phi, dphi)` of vectorised NumPy functions. Names: `sigmoid`, `tanh`, `relu`, `leaky_relu` ($\alpha=0.01$),
`elu` ($\alpha=1$), `gelu` (exact, via `scipy.special.erf`), `swish`. Each derivative is checked by finite differences away from $z=0$.

In [ ]:
ACT = {
    # 'relu': (lambda z: ..., lambda z: ...),
}

_z = np.concatenate([np.linspace(-6, -0.05, 60), np.linspace(0.05, 6, 60)])
_ref = {'sigmoid': expit, 'tanh': np.tanh, 'relu': lambda z: np.maximum(0, z), 'leaky_relu': lambda z: np.where(z > 0, z, 0.01 * z),
        'elu': lambda z: np.where(z > 0, z, np.expm1(z)), 'gelu': lambda z: z * _norm.cdf(z), 'swish': lambda z: z * expit(z)}
for _name, _f in _ref.items():
    if _name not in ACT:
        check(_name, None, 0); continue
    _phi, _dphi = ACT[_name]
    check(_name + ' value', _phi(_z), _f(_z))
    check(_name + ' derivative', _dphi(_z), _fd(_f, _z), tol=1e-6)

<details>
<summary><b>💡 Hint</b></summary>

GELU: $\Phi(z) = \tfrac12(1 + \mathrm{erf}(z/\sqrt2))$ and $\mathrm{GELU}'(z) = \Phi(z) + z\varphi(z)$ with $\varphi(z) = e^{-z^2/2}/\sqrt{2\pi}$. ELU': $1$ for $z>0$, $e^z$ otherwise.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Note how several derivatives reuse the forward value ($\sigma(1-\sigma)$, $1-\tanh^2$, ELU' $= \phi(z) + 1$ for $z<0$): frameworks cache outputs for exactly this reason.
At $z=0$ ReLU is not differentiable; frameworks simply use $\phi'(0) = 0$ (a valid subgradient).

```python
_Phi = lambda z: 0.5 * (1 + erf(z / np.sqrt(2)))
_pdf = lambda z: np.exp(-z ** 2 / 2) / np.sqrt(2 * np.pi)
ACT = {
    'sigmoid': (expit, lambda z: expit(z) * (1 - expit(z))),
    'tanh': (np.tanh, lambda z: 1 - np.tanh(z) ** 2),
    'relu': (lambda z: np.maximum(0, z), lambda z: (z > 0).astype(float)),
    'leaky_relu': (lambda z: np.where(z > 0, z, 0.01 * z), lambda z: np.where(z > 0, 1.0, 0.01)),
    'elu': (lambda z: np.where(z > 0, z, np.expm1(np.minimum(z, 0))), lambda z: np.where(z > 0, 1.0, np.exp(np.minimum(z, 0)))),
    'gelu': (lambda z: z * _Phi(z), lambda z: _Phi(z) + z * _pdf(z)),
    'swish': (lambda z: z * expit(z), lambda z: expit(z) + z * expit(z) * (1 - expit(z))),
}
```

</details>

### Exercise 11 · A numerically stable sigmoid
*💻 Code · ★☆☆*

The naive `1 / (1 + np.exp(-z))` overflows for $z = -1000$ (a `RuntimeWarning`). Implement `stable_sigmoid(z)` that never evaluates
`exp` of a large positive number, by using $\sigma(z) = \frac{e^{z}}{1+e^{z}}$ for $z<0$. Also implement `log_sigmoid(z)` $=\log\sigma(z)$ stably
(needed for BCE with logits).

In [ ]:
def stable_sigmoid(z):
    return None

def log_sigmoid(z):
    return None

z_ext = np.array([-1000., -30, -1, 0, 1, 30, 1000])

with warnings.catch_warnings():
    warnings.simplefilter('error')   # any overflow warning becomes an error
    try:
        _s = stable_sigmoid(z_ext); _ls = log_sigmoid(z_ext)
    except RuntimeWarning as e:
        print('❌ overflow warning:', e); _s = _ls = 'overflow'
check('stable sigmoid', _s, expit(z_ext))
check('log sigmoid', _ls, -np.logaddexp(0, -z_ext))

<details>
<summary><b>💡 Hint</b></summary>

`np.where` evaluates both branches, so compute `e = np.exp(-np.abs(z))` (always ≤ 1) and build both formulas from it. $\log\sigma(z) = -\log(1+e^{-z}) = \min(z,0) - \log(1 + e^{-|z|})$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

With $e = e^{-|z|}\in(0,1]$: $\sigma(z) = 1/(1+e)$ for $z\ge0$ and $e/(1+e)$ for $z<0$. For the log: $\log\sigma(z) = \min(z,0) - \mathrm{log1p}(e^{-|z|})$,
which gives $-1000$ at $z=-1000$ instead of $\log 0 = -\infty$. That is why losses should take **logits** (e.g. `BCEWithLogitsLoss`).

```python
def stable_sigmoid(z):
    z = np.asarray(z, float)
    e = np.exp(-np.abs(z))
    return np.where(z >= 0, 1 / (1 + e), e / (1 + e))

def log_sigmoid(z):
    z = np.asarray(z, float)
    return np.minimum(z, 0) - np.log1p(np.exp(-np.abs(z)))

z_ext = np.array([-1000., -30, -1, 0, 1, 30, 1000])
```

</details>

### Exercise 12 · Softmax with temperature
*💻 Code · ★★☆*

Implement `softmax_T(z, T)` $=\mathrm{softmax}(z/T)$ (stable). For $z = (2, 1, 0.5, -1)$ compute the entropy $H(p) = -\sum p\log p$ for
$T \in \{0.1, 0.5, 1, 2, 10\}$ and store it in `entropies` (a list). What happens as $T\to0$ and $T\to\infty$? Where is this used?

In [ ]:
def softmax_T(z, T):
    return None

z_T = np.array([2., 1, 0.5, -1])
temps = [0.1, 0.5, 1, 2, 10]
entropies = None

check('softmax_T(z, 2)', softmax_T(z_T, 2.0), _sp_softmax(z_T / 2.0))
if entropies is None:
    check('entropy increases with T toward log 4', None, True)
else:
    check('entropy increases with T toward log 4', bool(np.all(np.diff(entropies) > 0) and entropies[0] < 0.01 and abs(entropies[-1] - np.log(4)) < 0.05), True)

<details>
<summary><b>💡 Hint</b></summary>

Entropy: `-(p * np.log(p + 1e-300)).sum()`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$T\to0$: the distribution becomes one-hot on the argmax (entropy $\to0$). $T\to\infty$: uniform (entropy $\to\log K = \log 4 \approx 1.386$).
Used for sampling from language models (low $T$ = conservative, high $T$ = diverse), knowledge distillation (soft targets at $T>1$),
and contrastive losses (InfoNCE temperature, [[Self-Supervised and Contrastive Learning]]).

```python
def softmax_T(z, T):
    s = np.asarray(z, float) / T
    e = np.exp(s - s.max())
    return e / e.sum()

z_T = np.array([2., 1, 0.5, -1])
temps = [0.1, 0.5, 1, 2, 10]
entropies = [float(-(p * np.log(p + 1e-300)).sum()) for p in (softmax_T(z_T, T) for T in temps)]
print(np.round(entropies, 4))
```

</details>

### Exercise 13 · Counting dead ReLUs
*💻 Code · ★★☆*

Simulate a layer after a bad update: 200 hidden units, $W\sim\mathcal N(0, 2/d)$, $d=20$, but biases $b\sim\mathcal N(-3, 1)$.
For 1000 inputs $x\sim\mathcal N(0, I)$ compute the fraction of units that are **dead** (output 0 for every input) with ReLU, and the fraction
of units with zero gradient for every input under Leaky ReLU. Store them in `dead_relu` and `dead_leaky`.

In [ ]:
_r = np.random.default_rng(5)
d, n_units = 20, 200
W_dead = _r.normal(0, np.sqrt(2 / d), (d, n_units))
b_dead = _r.normal(-3, 1, n_units)
X_dead = _r.normal(size=(1000, d))
dead_relu = None
dead_leaky = None

_Z = X_dead @ W_dead + b_dead
check('dead fraction (ReLU)', dead_relu, np.mean(np.all(np.maximum(0, _Z) == 0, axis=0)))
check('dead fraction (Leaky)', dead_leaky, 0.0)

<details>
<summary><b>💡 Hint</b></summary>

A unit is dead if `(Z <= 0).all(axis=0)`. Each $z_j \sim\mathcal N(b_j, \lVert w_j\rVert^2)$ with $\lVert w_j\rVert^2\approx 2$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

With $b\approx-3$ and pre-activation std $\approx\sqrt2$, most units are negative for *most* inputs, and those with $b_j \lesssim -4$ are negative for **all** 1000:
about 13% of the units are dead here (the exact number depends on the draw); many more are active only on a handful of inputs. With Leaky ReLU the gradient is $\alpha=0.01 \ne 0$ everywhere, so **no** unit is dead: it can recover, just slowly.

```python
_r = np.random.default_rng(5)
d, n_units = 20, 200
W_dead = _r.normal(0, np.sqrt(2 / d), (d, n_units))
b_dead = _r.normal(-3, 1, n_units)
X_dead = _r.normal(size=(1000, d))
Z = X_dead @ W_dead + b_dead
dead_relu = np.mean((Z <= 0).all(axis=0))
leaky_grad = np.where(Z > 0, 1.0, 0.01)
dead_leaky = np.mean((leaky_grad == 0).all(axis=0))
print(dead_relu, dead_leaky)
```

</details>

### Exercise 14 · Gradient flow through depth, per activation
*💻 Code · ★★★*

For each of `sigmoid`, `tanh`, `relu`, `gelu` build a 30-layer MLP of width 100 with init $W\sim\mathcal N(0, g/n_{in})$, where $g = 2$ for ReLU/GELU and $g = 1$ otherwise.
Feed a batch of 256 standard-normal inputs, set the upstream gradient at the top to all ones/$N$, backpropagate **through the activations only**
(no loss) and record $\lVert \partial L/\partial h^{(0)}\rVert$ (gradient w.r.t. the network input) in a dict `input_grad`.
Rank the activations. Use your `ACT` dictionary from Exercise 10.

In [ ]:
input_grad = None   # {'sigmoid': ..., 'tanh': ..., 'relu': ..., 'gelu': ...}

if input_grad is None:
    check('sigmoid gradient vanishes far below the others', None, True)
else:
    print({k: '%.2e' % v for k, v in input_grad.items()})
    check('sigmoid gradient vanishes far below the others', input_grad['sigmoid'] < 1e-6 * min(input_grad['tanh'], input_grad['relu'], input_grad['gelu']), True)

<details>
<summary><b>💡 Hint</b></summary>

Forward: cache every pre-activation `Z_l`. Backward: `D = (D * dphi(Z_l)) @ W_l.T` from the top layer down.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Sigmoid's input gradient is about 18 orders of magnitude smaller than the others: each layer multiplies by $\le 0.25$ (and its non-zero-centred outputs shift
later pre-activations). Tanh with Xavier init and ReLU/GELU with He init keep the norm within one or two orders of magnitude over 30 layers.
The lesson in the note: the **activation and the initialisation must be chosen together** (He for ReLU, Xavier for tanh, [[Training Tricks]]).

```python
input_grad = {}
for name in ['sigmoid', 'tanh', 'relu', 'gelu']:
    phi, dphi = ACT[name]
    g = 2.0 if name in ('relu', 'gelu') else 1.0
    r = np.random.default_rng(0)
    H = r.normal(size=(256, 100))
    Ws, Zs = [], []
    for _ in range(30):
        W = r.normal(0, np.sqrt(g / 100), (100, 100))
        Z = H @ W
        Ws.append(W); Zs.append(Z)
        H = phi(Z)
    D = np.ones_like(H) / 256
    for W, Z in zip(reversed(Ws), reversed(Zs)):
        D = (D * dphi(Z)) @ W.T
    input_grad[name] = np.linalg.norm(D)
print(input_grad)
```

</details>

### Exercise 15 · SELU is self-normalising
*💻 Code · ★★★*

SELU is $\lambda z$ for $z>0$ and $\lambda\alpha(e^z - 1)$ otherwise, with $\lambda\approx1.0507$, $\alpha\approx1.6733$.
Propagate 2000 standard-normal inputs (width 256) through 50 layers with **LeCun-normal** init $W\sim\mathcal N(0, 1/n_{in})$ and no biases,
using SELU, tanh and ReLU. Store the mean and standard deviation of the last layer's activations in `stats` (dict: name → `(mean, std)`).

In [ ]:
SELU_L, SELU_A = 1.0507009873554805, 1.6732632423543772
def selu(z):
    return None

stats = None

check('selu', selu(np.array([-2., 0.5])), [SELU_L * SELU_A * np.expm1(-2.), SELU_L * 0.5])
if stats is None:
    check('SELU keeps mean ~ 0 and std ~ 1', None, True)
else:
    print({k: tuple(round(float(x), 3) for x in v) for k, v in stats.items()})
    _m, _s = stats['selu']
    check('SELU keeps mean ~ 0 and std ~ 1', abs(_m) < 0.15 and abs(_s - 1) < 0.15 and abs(stats['relu'][1] - 1) > 0.5, True)

<details>
<summary><b>💡 Hint</b></summary>

`np.where(z > 0, SELU_L * z, SELU_L * SELU_A * np.expm1(np.minimum(z, 0)))`. Use one fresh weight matrix per layer.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The SELU constants are chosen so that $(\mu, \sigma^2) = (0, 1)$ is a stable fixed point of the layer map under LeCun init: the last layer still has
mean $\approx0$, std $\approx1$. Tanh's std shrinks slowly (toward 0) and ReLU with this init halves the second moment every layer, so its std collapses toward 0
(He init would fix ReLU's scale, but not its non-zero mean). SELU achieves "normalisation" without a BatchNorm layer, but only for plain fully connected nets.

```python
SELU_L, SELU_A = 1.0507009873554805, 1.6732632423543772
def selu(z):
    return np.where(z > 0, SELU_L * z, SELU_L * SELU_A * np.expm1(np.minimum(z, 0)))

stats = {}
for name, phi in [('selu', selu), ('tanh', np.tanh), ('relu', lambda z: np.maximum(0, z))]:
    r = np.random.default_rng(1)
    H = r.normal(size=(2000, 256))
    for _ in range(50):
        H = phi(H @ r.normal(0, np.sqrt(1 / 256), (256, 256)))
    stats[name] = (H.mean(), H.std())
print(stats)
```

</details>

---
*Done? Revisit the exercises you needed hints for in a few days — retrieval beats rereading. Back to [Activation Functions](Activation%20Functions.md).*